# **Project Name**    - Brain Tumor




##### **Project Type**    - Classification
##### **Contribution**    - Individual
##### **P.Sandeep**


# **Project Summary -**

Brain tumors present one of the most critical clinical challenges in contemporary oncology, where early, precise, and non-invasive detection is vital for effective therapeutic interventions and improved patient prognosis. Conventional diagnosis relies heavily on magnetic resonance imaging (MRI) evaluated by radiologists. However, manual visual assessment is inherently labor-intensive, time-consuming, and subject to inter-observer variability, particularly across structurally subtle pathologies. This project delivers an automated, deep-learning-powered diagnostic framework for multi-class brain tumor classification across four distinct diagnostic classes: Glioma, Meningioma, Pituitary tumor, and No Tumor.   The neuroimaging dataset comprises a total of 2,443 brain MRI scans organized into standardized splits: a Training split of 1,695 scans (69.38%), a Validation split of 502 scans (20.55%), and a Test split of 246 scans (10.07%). The distribution includes 805 Glioma, 610 Pituitary, 545 Meningioma, and 483 No Tumor scans. Exploratory Data Analysis (EDA) and metadata extraction revealed that raw scans exhibited high integrity with zero missing, corrupt, or duplicate samples, alongside uniform aspect ratios centered tightly at 1.0, ensuring minimal geometric distortion during spatial standardization. Furthermore, statistical hypothesis testing via One-Way ANOVA and Welch’s t-test verified statistically significant distinctions in tissue grayscale intensity and contrast variances between pathological and healthy brain parenchyma, establishing a robust quantitative foundation for deep learning classification.   To address moderate class imbalance and real-world imaging discrepancies such as patient head tilt, an automated preprocessing and real-time data augmentation pipeline was implemented, incorporating dynamic spatial rotations, horizontal flipping, zooming, and min-max pixel intensity rescaling to $[0.0, 1.0]$ across standardized $224 \times 224 \times 3$ image dimensions.   Model development followed an iterative benchmarking strategy comparing three distinct deep learning architectures: a Custom Deep Convolutional Neural Network (CNN) built from scratch featuring progressive feature extraction blocks with Batch Normalization and Dropout; a lightweight MobileNetV2 transfer learning backbone; and a compound-scaled EfficientNetB0 architecture. Training pipelines leveraged Adam optimization, dynamic learning rate scheduling via ReduceLROnPlateau, balanced class weighting, and early stopping callbacks to prevent overfitting and ensure numerical stability.   Comprehensive model evaluation on the hold-out test set prioritized clinical sensitivity (Recall) and Macro F1-score alongside categorical accuracy, mitigating potential majority-class prediction bias. While the custom baseline architecture established a competitive benchmark, transfer learning backbones achieved superior feature abstraction, minimizing false negative diagnostic oversights across minority categories. Model interpretability was incorporated through Gradient-Weighted Class Activation Mapping (Grad-CAM), generating coarse visual heatmaps that confirm model decision boundaries localize to authentic anatomical lesion masses and peritumoral edema rather than irrelevant background air margins or skull artifacts.   Finally, the end-to-end framework was packaged for operational clinical triage support by serializing trained models into production .h5 and .keras formats and deploying an interactive, multi-model Streamlit web dashboard in Visual Studio Code. The application enables healthcare practitioners to seamlessly upload unknown brain MRI scans, toggle across architectures, review real-time diagnostic predictions with multi-class confidence distributions, and access lesion context, providing an efficient, reliable decision-support system for medical imaging workflows

# **GitHub Link -**

https://github.com/sandeeppasupula/Brain-Tumor-MRI-Classification

# **Problem Statement**


Brain tumors represent life-threatening central nervous system neoplasms that demand prompt, accurate, and non-invasive diagnosis to optimize surgical planning and patient prognosis. While Magnetic Resonance Imaging (MRI) serves as the standard clinical modality for neurological examination, conventional manual evaluation by radiologists is labor-intensive, subject to cognitive fatigue, and susceptible to inter-observer variability—particularly when distinguishing between subtle morphological margins of different tumor pathologies.   The objective of this project is to develop and deploy an automated, end-to-end deep learning classification system capable of accurately categorizing brain MRI scans into four distinct diagnostic classes: Glioma, Meningioma, Pituitary tumor, and No Tumor. The pipeline must resolve class imbalances and scanner variability through standardized preprocessing and augmentation, benchmark a Custom CNN against state-of-the-art Transfer Learning architectures (MobileNetV2 and EfficientNetB0), ensure clinical interpretability via Grad-CAM visual saliency heatmaps, and integrate into an interactive Streamlit web application for real-time diagnostic decision support.   

# ***Let's Begin !***

In [ ]:
#To access the files which are present in Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#To Extract folders in Brain Tumor folder in Drive
from google.colab import drive
import zipfile
import os

# 1. Mount Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Check the Brain Tumor folder
possible_folder = '/content/drive/MyDrive/Brain Tumor'

if not os.path.exists(possible_folder):
    # Search MyDrive if the exact folder name differs
    print("Folder '/content/drive/MyDrive/Brain Tumor' not found directly.")
    print("Available folders:", [f for f in os.listdir('/content/drive/MyDrive') if not f.startswith('.')])
else:
    print("Found 'Brain Tumor' folder! Contents:")
    folder_contents = os.listdir(possible_folder)
    print(folder_contents)

    # 3. Setup extraction directory on Colab fast disk
    raw_extract_dir = '/content/dataset_raw'
    os.makedirs(raw_extract_dir, exist_ok=True)

    # 4. Extract any zip files found inside the folder
    for fname in folder_contents:
        if fname.lower().endswith('.zip'):
            z_path = os.path.join(possible_folder, fname)
            split_name = os.path.splitext(fname)[0].lower()
            dest = os.path.join(raw_extract_dir, split_name)
            os.makedirs(dest, exist_ok=True)

            print(f"Extracting {fname} to {dest}...")
            with zipfile.ZipFile(z_path, 'r') as zip_ref:
                zip_ref.extractall(dest)

    print("\nExtraction completed successfully!")

In [ ]:
#To know the number of images in each class
import os

target_classes = {'glioma', 'meningioma', 'no_tumor', 'pituitary'}

def find_split_dir(keyword):
    """Finds the directory containing the class subfolders for train, valid, or test."""
    for root, dirs, _ in os.walk('/content/dataset_raw'):
        # Check if the path belongs to the requested split
        if keyword.lower() in root.lower():
            dir_set = {d.lower() for d in dirs}
            # Check if at least 3 of our classes are inside this folder
            if len(target_classes.intersection(dir_set)) >= 3:
                return root
    return None

train_dir = find_split_dir('train')
valid_dir = find_split_dir('valid')
test_dir  = find_split_dir('test')

print("Resolved Directories:")
print("Train path:", train_dir)
print("Valid path:", valid_dir)
print("Test path: ", test_dir)

# Print count of images per class
splits = {'Train': train_dir, 'Valid': valid_dir, 'Test': test_dir}
for split_name, split_path in splits.items():
    print(f"\n--- {split_name} Split ---")
    if split_path and os.path.exists(split_path):
        for cls in sorted(os.listdir(split_path)):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                img_count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                print(f"  {cls}: {img_count} images")
    else:
        print("  Directory not found!")

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

### Dataset Loading

In [ ]:
# Load Dataset
# Define the paths to our three splits (verified from previous steps)
split_directories = {
    'train': train_dir,
    'valid': valid_dir,
    'test':  test_dir
}

# Empty list where we will collect information about every single image
dataset_records = []

# Loop through each split ('train', 'valid', 'test')
for split_name, split_path in split_directories.items():

    # Loop through each tumor category folder ('glioma', 'meningioma', 'no_tumor', 'pituitary')
    for class_name in sorted(os.listdir(split_path)):
        class_folder = os.path.join(split_path, class_name)

        # Check if it is indeed a folder
        if os.path.isdir(class_folder):

            # Read every image file inside the folder
            for file_name in os.listdir(class_folder):
                if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    full_image_path = os.path.join(class_folder, file_name)

                    try:
                        # 1. Open image using PIL to get width and height
                        with Image.open(full_image_path) as img:
                            width, height = img.size
                            channels = len(img.getbands())

                        # 2. Open image in grayscale using OpenCV to calculate brightness and contrast
                        gray_img = cv2.imread(full_image_path, cv2.IMREAD_GRAYSCALE)
                        mean_pixel_value = float(np.mean(gray_img))
                        std_pixel_value = float(np.std(gray_img))

                        # 3. Append dictionary record to list
                        dataset_records.append({
                            'filename': file_name,
                            'split': split_name,
                            'class': class_name,
                            'width': width,
                            'height': height,
                            'aspect_ratio': round(width / height, 4),
                            'channels': channels,
                            'mean_intensity': round(mean_pixel_value, 2),
                            'std_intensity': round(std_pixel_value, 2),
                            'filepath': full_image_path
                        })
                    except Exception as error:
                        print(f"Could not read image {file_name}: {error}")

# Convert our list of records into a Pandas DataFrame
df = pd.DataFrame(dataset_records)

print("Dataset loading complete!")
print(f"Total images cataloged: {len(df)}")

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
total_rows = df.shape[0]
total_columns = df.shape[1]

print(f"Total Rows (Total Number of MRI Scans): {total_rows}")
print(f"Total Columns (Features Extracted): {total_columns}")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_files = df['filename'].duplicated().sum()
duplicate_rows = df.duplicated().sum()

print(f"Duplicate Filenames Count: {duplicate_files}")
print(f"Complete Duplicate Rows Count: {duplicate_rows}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(9, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='Blues')
plt.title("Missing Values Heatmap (Dataset is 100% Clean)", fontsize=13)
plt.xlabel("Columns")
plt.ylabel("Image Index")
plt.show()

### What did you know about your dataset?

Dataset Overview: A multi-class neuroimaging dataset consisting of 2,443 brain MRI scans categorized into four classes: Glioma, Meningioma, Pituitary, and No Tumor.

Pre-Partitioned Splits: Divided into Train (1,695 images, ~69.4%), Validation (502 images, ~20.6%), and Test (246 images, ~10.1%).

Class Distribution: The overall distribution is 805 Glioma (32.9%), 610 Pituitary (25.0%), 545 Meningioma (22.3%), and 483 No Tumor (19.8%).

Key Observations:

Zero missing, null, or corrupted files exist across the dataset.

Glioma is the most prevalent category, requiring data augmentation and metrics like Precision, Recall, and F1-score to counteract class imbalance.

Scans require resizing to $224 \times 224 \times 3$ and pixel normalization $[0, 1]$ before passing into CNN models

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns.tolist()

In [ ]:
# Dataset Describe
df.describe().T

### Variables Description

filename: Unique identifier of the individual brain MRI image slice.

split: Dataset partition category indicating usage (train, valid, test).   

class: Diagnostic tumor target label (glioma, meningioma, no_tumor, pituitary).

width & height: Spatial dimensions (pixel resolution) of the raw MRI scan.   

aspect_ratio: Ratio of image width to height ($\text{width} / \text{height}$), used to check for distortion risks prior to resizing.

channels: Number of color channels in the image matrix ($1$ for grayscale, $3$ for RGB).

mean_intensity: Average grayscale pixel intensity value ($0–255$), representing overall scan brightness and exposure.

std_intensity: Standard deviation of pixel intensities, indicating contrast levels and tissue texture variation.

filepath: Absolute file system path used by data loaders to read the scan tensors into the deep learning pipeline.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in df.columns:
    print(f"Unique values in '{col}': {df[col].nunique()}")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# Step 1: Make a working copy of our metadata DataFrame
df_clean = df.copy()

# Step 2: Standardize text columns (strip whitespace & lowercase class labels)
df_clean['class'] = df_clean['class'].astype(str).str.strip().str.lower()
df_clean['split'] = df_clean['split'].astype(str).str.strip().str.lower()

# Step 3: Verify and enforce correct data types for image attributes
df_clean['width'] = df_clean['width'].astype(int)
df_clean['height'] = df_clean['height'].astype(int)
df_clean['aspect_ratio'] = df_clean['aspect_ratio'].astype(float)
df_clean['mean_intensity'] = df_clean['mean_intensity'].astype(float)
df_clean['std_intensity'] = df_clean['std_intensity'].astype(float)

# Step 4: Feature Engineering - Add Image Area and Orientation categories
df_clean['total_pixels'] = df_clean['width'] * df_clean['height']

# Classify image orientation based on aspect ratio
def assign_orientation(ratio):
    if 0.98 <= ratio <= 1.02:
        return 'Square'
    elif ratio > 1.02:
        return 'Landscape'
    else:
        return 'Portrait'

df_clean['orientation'] = df_clean['aspect_ratio'].apply(assign_orientation)

# Step 5: Map string target classes to numeric integer labels for modeling
class_mapping = {
    'glioma': 0,
    'meningioma': 1,
    'no_tumor': 2,
    'pituitary': 3
}
df_clean['class_id'] = df_clean['class'].map(class_mapping)

# Step 6: Validate cleaned dataset
print("Data Wrangling Completed Successfully!")
print(f"Cleaned Dataset Shape: {df_clean.shape}")
print("\nOrientation Breakdown:")
print(df_clean['orientation'].value_counts())
print("\nClass ID Mapping Check:")
print(df_clean[['class', 'class_id']].drop_duplicates().to_string(index=False))

# Update working dataframe
df = df_clean

### What all manipulations have you done and insights you found?

**Manipulations Performed:**

Directory-to-Tabular Extraction: Iterated through all nested image folders across train, valid, and test splits, parsing 2,443 image files into structured tabular metadata containing image resolution, channels, and pixel statistics.   

Text Standardization: Stripped unintended leading/trailing whitespace and normalized class labels to lowercase to prevent categorical mismatch errors.   

Data Type Casting: Enforced integer formats for spatial dimensions (width, height, total_pixels) and float precision for calculated intensity statistics (mean_intensity, std_intensity, aspect_ratio).

Feature Engineering:Created total_pixels to measure scan spatial capacity.Derived orientation (Square, Landscape, Portrait) using the aspect ratio to evaluate geometric distortion risk prior to $224 \times 224$ resizing.   Created class_id mapping tumor classes (glioma: 0, meningioma: 1, no_tumor: 2, pituitary: 3) to numeric identifiers for deep learning loss functions.   

Corrupt File Validation: Validated that all 2,443 files load successfully through PIL and OpenCV without format corruption or missing values.

**Key Insights Found:**

Zero Corrupted/Null Records: The entire image dataset of 2,443 scans is fully accessible with 0 missing pixel arrays or broken headers.

Aspect Ratio Consistency: The vast majority of MRI scans share an aspect ratio near $1.0$ (square geometry), confirming that resizing images directly to the target $224 \times 224 \times 3$ dimension will introduce minimal anatomical shape distortion.

Class Imbalance in Training Split: The training split contains 564 Glioma scans ($33.3\%$) compared to 335 No Tumor scans ($19.8\%$). Because Glioma is the dominant class, data augmentation and balanced metrics (Macro F1-score, Precision, Recall) will be essential to prevent model bias.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
#Chart-1 Visualizaation: Overall Class Distribution
plt.figure(figsize=(8, 5))
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Count plot of tumor classes across the dataset
ax = sns.countplot(data=df, x='class', palette=palette, order=sorted(df['class'].unique()))
plt.title("Total Image Count per Tumor Category", fontsize=14, weight='bold')
plt.xlabel("Diagnostic Class", fontsize=11)
plt.ylabel("Total Scan Count", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate counts on top of bars
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')


plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A count bar chart is the most direct univariate visualization to assess the frequency and balance of discrete target categories in classification tasks.

##### 2. What is/are the insight(s) found from the chart?

The dataset contains 805 Glioma, 610 Pituitary, 545 Meningioma, and 483 No Tumor scans. Glioma is the most prevalent class, comprising nearly 33% of the dataset, while No Tumor is the least frequent.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Identifying this imbalance ensures we do not rely purely on accuracy, guiding us to apply data augmentation and track Macro F1-score and Recall.

Risk of Inaction: Training without awareness of this distribution could bias the model to over-predict Glioma, resulting in misdiagnosed meningiomas or missed tumors (false negatives), which harms clinical trust and patient safety.  

#### Chart - 2

In [ ]:
# Chart - 2 visualization code# Chart - 2 visualization code: Distribution across Splits
plt.figure(figsize=(9, 5))
split_palette = {'train': '#3470a3', 'valid': '#f58518', 'test': '#54a24b'}

ax = sns.countplot(data=df, x='class', hue='split', palette=split_palette, order=sorted(df['class'].unique()))
plt.title("Class Balance Across Train, Valid, and Test Partitions", fontsize=14, weight='bold')
plt.xlabel("Tumor Class", fontsize=11)
plt.ylabel("Number of Scans", fontsize=11)
plt.legend(title="Data Split")
plt.grid(axis='y', linestyle='--', alpha=0.5)

for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{int(h)}", (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A grouped bar chart allows immediate visual comparison of multi-class sub-distributions across independent partitions (train, valid, test).

##### 2. What is/are the insight(s) found from the chart?

The stratified proportion is consistent: Train represents ~69.4% (1,695), Valid represents ~20.6% (502), and Test represents ~10.1% (246) across all classes

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Consistent split ratios ensure that validation and test loss accurately mirror real-world training generalization, avoiding distribution shift.

Risk of Inaction: If the test set lacked representative tumor subtypes, reported evaluation metrics would be overly optimistic, failing upon live hospital deployment.   

#### Chart - 3

In [ ]:
# Chart - 3 visualization code: Visual Scan Inspection Grid
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
classes = sorted(df['class'].unique())

for idx, cls in enumerate(classes):
    sample_path = df[df['class'] == cls]['filepath'].iloc[0]
    img = cv2.imread(sample_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"Class: {cls.upper()}", fontsize=12, weight='bold')
    axes[idx].axis('off')

plt.suptitle("Representative Axial MRI Slices per Class", fontsize=15, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Direct image grid visualization is necessary in computer vision to inspect raw pixel clarity, contrast, and anatomical lesion presentation.

##### 2. What is/are the insight(s) found from the chart?

Tumor types exhibit distinct morphological signatures: Pituitary tumors concentrate near the sella turcica, Meningiomas attach near brain membranes/meninges, Gliomas show infiltrative patterns, and No Tumor scans display symmetric brain anatomy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms that distinct visual structural patterns exist for a Convolutional Neural Network to extract.

Risk of Inaction: Deploying deep models without visually verifying raw input scans risks training on corrupted images or non-brain artifacts.   

#### Chart - 4

In [ ]:
# Chart - 4 visualization code: Width vs. Height Scatter Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='width', y='height', hue='class', alpha=0.7, s=50, palette='Set2')
plt.title("Raw MRI Resolution (Width vs. Height)", fontsize=14, weight='bold')
plt.xlabel("Width (Pixels)", fontsize=11)
plt.ylabel("Height (Pixels)", fontsize=11)
plt.axvline(224, color='red', linestyle='--', label='Target Resize (224px)')
plt.axhline(224, color='red', linestyle='--')
plt.legend(title="Class")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A bivariate scatter plot reveals resolution clusters, aspect ratio consistency, and dimension variation among images from different MRI machines.

##### 2. What is/are the insight(s) found from the chart?

Most scans cluster along the diagonal (equal width and height), though original dimensions range widely, confirming input size heterogeneity.   

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Validates the standard preprocessing step of resizing all images uniformly to $224 \times 224$ pixels for CNN architectures.

Risk of Inaction: Passing raw, variable-sized matrices directly to deep learning layers causes tensor dimension mismatches and runtime crashes.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code: Aspect Ratio Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['aspect_ratio'], bins=30, kde=True, color='teal')
plt.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Square Ratio (1.0)')
plt.title("Distribution of Scan Aspect Ratios (Width / Height)", fontsize=14, weight='bold')
plt.xlabel("Aspect Ratio", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with Kernel Density Estimation (KDE) highlights how closely image geometry aligns with a 1:1 square ratio.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of scans have aspect ratios tightly centered between 0.98 and 1.02, confirming that most scans are already square.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Resizing images directly to $224 \times 224$ will not distort anatomical structures or tumor margins.   Risk of Inaction: If images had extreme aspect ratios, naive resizing would stretch tissue, causing the model to learn distorted morphological patterns.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code: Mean Pixel Intensity Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='class', y='mean_intensity', palette='Pastel1')
plt.title("Mean Pixel Intensity Distribution Across Tumor Types", fontsize=14, weight='bold')
plt.xlabel("Tumor Class", fontsize=11)
plt.ylabel("Average Grayscale Pixel Intensity (0 - 255)", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot displays the median, interquartile range (IQR), and outliers of grayscale brightness across different tumor classes.

##### 2. What is/are the insight(s) found from the chart?

Tumor scans frequently exhibit slightly higher median brightness and variance than no_tumor scans due to hyperintense tumor mass tissue and edema.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms that tissue brightness carries diagnostic signal, reinforcing the need for min-max normalization ($[0, 1]$) so brightness scales consistently.

Risk of Inaction: Failing to normalize pixel values can lead to gradient explosion during backpropagation and cause the network to overfit to scanner brightness differences.   

#### Chart - 7

In [ ]:
# Chart - 7 visualization code: Pixel Intensity Contrast Violin Plot
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='class', y='std_intensity', palette='Set3', inner='quartile')
plt.title("Pixel Intensity Contrast Variance by Class", fontsize=14, weight='bold')
plt.xlabel("Tumor Class", fontsize=11)
plt.ylabel("Standard Deviation of Pixel Intensity", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A violin plot captures both the distribution density and spread of scan contrast (standard deviation of pixel intensity) across categories

##### 2. What is/are the insight(s) found from the chart?

Meningioma and Glioma scans exhibit broader contrast variance distributions, reflecting heterogeneous tissue density around tumor boundaries

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Shows that internal textural contrast is a discriminative property that convolutional kernels can leverage.

Risk of Inaction: Neglecting contrast adjustments during augmentation could cause the network to struggle with low-contrast hospital scans.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code: Average Image Per Class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
target_size = (150, 150)

for idx, cls in enumerate(classes):
    cls_files = df[df['class'] == cls]['filepath'].iloc[:40]
    accumulated_img = np.zeros(target_size, dtype=np.float32)

    for f in cls_files:
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        im_resized = cv2.resize(im, target_size)
        accumulated_img += im_resized / len(cls_files)

    axes[idx].imshow(accumulated_img, cmap='inferno')
    axes[idx].set_title(f"Mean Map: {cls.capitalize()}", fontsize=12, weight='bold')
    axes[idx].axis('off')

plt.suptitle("Average Spatial Intensity Maps (Class Central Tendencies)", fontsize=15, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Composite heatmaps aggregate multiple scans to reveal the typical spatial regions where intensity differences and lesions concentrate.

##### 2. What is/are the insight(s) found from the chart?

Pituitary averages highlight central-base intensity peaks, whereas No Tumor displays bilateral ventricular symmetry

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Demonstrates that spatial anatomical location is informative, supporting the use of CNN architectures with translation-preserving spatial layers.

Risk of Inaction: Applying excessive spatial crop augmentation could remove key anatomical landmarks like the brain boundary, degrading model accuracy

#### Chart - 9

In [ ]:
# Chart - 9 visualization code: Subtraction Difference Maps
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
tumor_classes = ['glioma', 'meningioma', 'pituitary']

# Calculate base No-Tumor average
no_tumor_files = df[df['class'] == 'no_tumor']['filepath'].iloc[:40]
base_no_tumor = np.zeros((150, 150), dtype=np.float32)
for f in no_tumor_files:
    base_no_tumor += cv2.resize(cv2.imread(f, cv2.IMREAD_GRAYSCALE), (150, 150)) / len(no_tumor_files)

for idx, cls in enumerate(tumor_classes):
    cls_files = df[df['class'] == cls]['filepath'].iloc[:40]
    cls_mean = np.zeros((150, 150), dtype=np.float32)
    for f in cls_files:
        cls_mean += cv2.resize(cv2.imread(f, cv2.IMREAD_GRAYSCALE), (150, 150)) / len(cls_files)

    diff = np.abs(cls_mean - base_no_tumor)
    axes[idx].imshow(diff, cmap='hot')
    axes[idx].set_title(f"{cls.capitalize()} minus No-Tumor", fontsize=12, weight='bold')
    axes[idx].axis('off')

plt.suptitle("Structural Anomaly Subtraction Maps", fontsize=15, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A subtraction difference map directly visualizes where pathological tissue deviates from healthy brain baseline anatomy.   

##### 2. What is/are the insight(s) found from the chart?

Highlight regions confirm distinct lesion zones: pituitary tumors create high variance in the inferior sellar area, while gliomas show dispersed supratentorial deviations.   

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Clinically verifies that target differences are localized and genuine, rather than scanner background noise.

Risk of Inaction: Training on images without background mask verification risks models latching onto scanner label artifacts rather than actual brain tissue.   

#### Chart - 10

In [ ]:
# Chart - 10 visualization code: Grayscale Density KDE
plt.figure(figsize=(9, 5))
for cls in sorted(df['class'].unique()):
    sample_files = df[df['class'] == cls]['filepath'].iloc[:15]
    all_pixels = []
    for f in sample_files:
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        all_pixels.extend(im.flatten()[::10]) # sample every 10th pixel for speed

    sns.kdeplot(all_pixels, label=cls.capitalize(), linewidth=2)

plt.title("Pixel Intensity Density Profiles Across Categories", fontsize=14, weight='bold')
plt.xlabel("Voxel Grayscale Value (0: Black background, 255: Bright tissue)", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.xlim(0, 255)
plt.legend(title="Class")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

KDE density curves show the full spectrum distribution of voxel intensities, distinguishing dark background air from dense brain matter and lesions.

##### 2. What is/are the insight(s) found from the chart?

All scans have a dominant spike near 0 (the black background around the skull) followed by a bimodal curve representing brain parenchyma and hyperintense tumor zones.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Shows that black background pixels are redundant, justifying data augmentation techniques like horizontal flipping and slight zooming to focus on brain tissue.

Risk of Inaction: Extreme brightness shifts during augmentation could wash out lower-intensity brain details, confusing meningiomas with healthy tissue.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code: Edge Complexity Boxplot
def compute_edges(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (150, 150))
    edges = cv2.Canny(img, 100, 200)
    return (edges > 0).sum()

# Compute edge counts for a sample subset
sample_df = df.groupby('class').apply(lambda s: s.sample(min(len(s), 40), random_state=42)).reset_index(drop=True)
sample_df['edge_count'] = sample_df['filepath'].apply(compute_edges)

plt.figure(figsize=(8, 5))
sns.boxplot(data=sample_df, x='class', y='edge_count', palette='Accent')
plt.title("Structural Edge Complexity (Canny Edge Pixel Counts)", fontsize=14, weight='bold')
plt.xlabel("Tumor Class", fontsize=11)
plt.ylabel("Total Detected Edge Pixels", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Canny edge detection quantifies structural boundary sharpness and anatomical complexity across slices.

##### 2. What is/are the insight(s) found from the chart?

Tumor categories (especially Glioma and Meningioma) exhibit higher edge pixel counts due to irregular tumor margins and peritumoral edema compared to healthy symmetrical slices.   

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms that shallow convolutional filters (which detect edges and borders) will extract useful discriminating features.

Risk of Inaction: Overly aggressive Gaussian blur augmentation could soften these critical diagnostic edge boundaries, reducing model sensitivity.   

#### Chart - 12

In [ ]:
# Chart - 12 visualization code: Data Augmentation Grid
sample_img_path = df['filepath'].iloc[0]
raw_sample = cv2.cvtColor(cv2.imread(sample_img_path), cv2.COLOR_BGR2RGB)
raw_sample = cv2.resize(raw_sample, (224, 224))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(raw_sample)
axes[0].set_title("1. Original Scan", fontsize=11, weight='bold')

# Horizontal Flip
axes[1].imshow(cv2.flip(raw_sample, 1))
axes[1].set_title("2. Horizontal Flip", fontsize=11, weight='bold')

# Slight Rotation
M = cv2.getRotationMatrix2D((112, 112), 15, 1.0)
rotated = cv2.warpAffine(raw_sample, M, (224, 224))
axes[2].imshow(rotated)
axes[2].set_title("3. 15° Rotation", fontsize=11, weight='bold')

# Brightness Shift
bright = np.clip(raw_sample.astype(np.int16) + 30, 0, 255).astype(np.uint8)
axes[3].imshow(bright)
axes[3].set_title("4. Brightness Boost", fontsize=11, weight='bold')

for ax in axes:
    ax.axis('off')

plt.suptitle("Visualizing Proposed Data Augmentation Transformations", fontsize=14, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A transformation comparison grid visually validates data augmentation steps before training to ensure they preserve diagnostic realism

##### 2. What is/are the insight(s) found from the chart?

Horizontal flips, 15-degree rotations, and slight brightness adjustments preserve brain anatomical structure while introducing orientation and contrast variety.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Augmentation addresses class imbalance and trains the CNN to recognize tumors regardless of patient head tilt, improving real-world clinical robustness.

Risk of Inaction: Extreme augmentations (e.g., vertical flips or heavy shearing) produce

#### Chart - 13

In [ ]:
# Chart - 13 visualization code: Scan Area by Orientation Category
plt.figure(figsize=(8, 5))
sns.stripplot(data=df, x='orientation', y='total_pixels', hue='class', jitter=0.25, alpha=0.7, palette='tab10')
plt.title("Scan Pixel Volume Across Geometric Orientations", fontsize=14, weight='bold')
plt.xlabel("Orientation Category", fontsize=11)
plt.ylabel("Total Pixel Area (Width x Height)", fontsize=11)
plt.legend(title="Class", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A jittered strip plot displays total image pixel volume across geometry groups (Square, Landscape, Portrait), showing distribution and density without obscuring individual scans.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of scans fall into the Square category across all four classes, with pixel areas clustering near typical scanner dimensions (e.g., $512 \times 512$).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms uniform data collection protocols across tumor types, showing no class-specific geometric bias.

Risk of Inaction: If a specific tumor class only appeared in non-square scans, the CNN could associate the aspect ratio itself with the disease, leading to shortcut learning rather than true lesion detection.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart - 14 visualization code: Correlation Heatmap

# 1. Select numeric features extracted from the MRI scans
numeric_features = ['width', 'height', 'aspect_ratio', 'mean_intensity', 'std_intensity', 'total_pixels']
corr_matrix = df[numeric_features].corr()

# 2. Plot correlation heatmap with annotations
plt.figure(figsize=(9, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)

plt.title("Correlation Heatmap of Extracted Image Attributes", fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap evaluates linear relationships and multicollinearity among all extracted continuous image metrics (spatial dimensions, pixel area, and tissue intensity statistics).   

##### 2. What is/are the insight(s) found from the chart?

width and height show a near-perfect positive correlation ($r \approx 1.00$), confirming that brain MRI scans maintain proportional aspect ratios.

mean_intensity and std_intensity display a moderate positive correlation ($r \approx 0.50$ to $0.65$), indicating that scans with brighter tissue regions also possess wider contrast variance.

Spatial dimensions (width, height) show virtually zero correlation ($r \approx 0.0$) with tissue brightness (mean_intensity), confirming that image resolution is independent of tissue contrast.

#### Chart - 15 - Pair Plot

In [ ]:
# Chart - 15 visualization code: Pair Plot
# Select a sample of scans to ensure the pair plot renders smoothly and quickly
sample_for_pairplot = df.groupby('class').apply(
    lambda x: x.sample(min(len(x), 50), random_state=42)
).reset_index(drop=True)

pairplot_features = ['aspect_ratio', 'mean_intensity', 'std_intensity', 'total_pixels', 'class']

# Generate pair plot colored by diagnostic tumor class
sns.pairplot(
    data=sample_for_pairplot[pairplot_features],
    hue='class',
    palette='tab10',
    diag_kind='kde',
    plot_kws={'alpha': 0.7, 's': 35}
)

plt.suptitle("Pairwise Relationships of MRI Metadata Features by Tumor Class", y=1.02, fontsize=15, weight='bold')
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot combines bivariate scatter plots with univariate kernel density distributions, making it possible to examine feature interactions and separation boundaries across the four tumor classes.

##### 2. What is/are the insight(s) found from the chart?

The diagonal density plots show that aspect_ratio is tightly distributed near $1.0$ across all four classes, confirming geometric uniformity.

The bivariate scatter plots between mean_intensity and std_intensity show noticeable overlap between classes, demonstrating that simple linear combinations of scalar image statistics cannot reliably separate tumor types.

Complex spatial features and non-linear patterns (such as lesion edges, textures, and anatomical positions) are required to distinguish the classes, justifying the need for deep convolutional networks.  

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Statement 1 (Pixel Intensity Across Pathologies): There is a statistically significant difference in mean grayscale pixel intensity across the four brain MRI categories (glioma, meningioma, no_tumor, pituitary), suggesting that distinct tumor tissue types alter overall scan luminance and tissue contrast.   


Statement 2 (Contrast Variance in Tumors vs. Healthy Brain): MRI scans diagnosed with active tumors (glioma, meningioma, pituitary) display significantly higher pixel intensity standard deviation (texture contrast variance) compared to healthy (no_tumor) brain scans due to hyperintense tumor mass and peritumoral edema.

Statement 3 (Geometric Aspect Ratio Independence): The distribution of scan aspect ratio categories (Square vs. Non-Square) is statistically independent of the diagnostic tumor classification, demonstrating that image geometric properties are not biased toward any particular tumor class

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis ($H_0$): There is no significant difference in the mean pixel intensity across the four diagnostic categories ($\mu_{\text{glioma}} = \mu_{\text{meningioma}} = \mu_{\text{no\_tumor}} = \mu_{\text{pituitary}}$).   

Alternative Hypothesis ($H_1$): At least one diagnostic category exhibits a mean pixel intensity that significantly differs from the others.   Significance Level ($\alpha$): $0.05$ (5%).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value for Hypothesis 1
import scipy.stats as stats

# 1. Group mean pixel intensity by tumor class
group_glioma = df[df['class'] == 'glioma']['mean_intensity']
group_meningioma = df[df['class'] == 'meningioma']['mean_intensity']
group_notumor = df[df['class'] == 'no_tumor']['mean_intensity']
group_pituitary = df[df['class'] == 'pituitary']['mean_intensity']

# 2. Perform One-Way ANOVA test
f_stat, p_value_1 = stats.f_oneway(group_glioma, group_meningioma, group_notumor, group_pituitary)

print("=" * 55)
print("HYPOTHESIS TEST 1: ONE-WAY ANOVA (MEAN PIXEL INTENSITY)")
print("=" * 55)
print(f"F-Statistic : {f_stat:.4f}")
print(f"P-Value     : {p_value_1:.4e}")

alpha = 0.05
if p_value_1 < alpha:
    print(f"Conclusion  : Reject the Null Hypothesis (p < {alpha}).")
    print("Insight     : Statistically significant intensity differences exist across classes.")
else:
    print(f"Conclusion  : Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Insight     : No significant difference in intensity across classes.")
print("=" * 55)

##### Which statistical test have you done to obtain P-Value?

One-Way Analysis of Variance (ANOVA).

##### Why did you choose the specific statistical test?

One-Way ANOVA is the standard parametric statistical test used to determine whether there are statistically significant differences among the means of three or more independent, mutually exclusive groups (in this case, four diagnostic tumor classes) on a continuous numerical dependent variable (mean_intensity). It tests whether across-group variance exceeds within-group variance without inflating the family-wise Type I error rate that would occur if multiple two-sample t-tests were conducted.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis ($H_0$): There is no significant difference in the mean contrast variance (std_intensity) between tumor-positive scans and healthy (no_tumor) scans ($\mu_{\text{tumor}} = \mu_{\text{no\_tumor}}$).

Alternative Hypothesis ($H_1$): Scans with brain tumors have a significantly higher contrast variance than healthy scans ($\mu_{\text{tumor}} > \mu_{\text{no\_tumor}}$).   Significance Level ($\alpha$): $0.05$ (5%).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Perform Statistical Test to obtain P-Value for Hypothesis 2
import scipy.stats as stats

# 1. Split into tumor-positive and healthy (no tumor) groups
tumor_std = df[df['class'].isin(['glioma', 'meningioma', 'pituitary'])]['std_intensity']
healthy_std = df[df['class'] == 'no_tumor']['std_intensity']

# 2. Perform Two-Sample Independent Welch's t-test (unequal variances assumed)
t_stat, p_value_2 = stats.ttest_ind(tumor_std, healthy_std, equal_var=False)

print("=" * 55)
print("HYPOTHESIS TEST 2: TWO-SAMPLE WELCH'S T-TEST")
print("=" * 55)
print(f"Tumor Scans Count   : {len(tumor_std)} (Mean std: {tumor_std.mean():.2f})")
print(f"Healthy Scans Count : {len(healthy_std)} (Mean std: {healthy_std.mean():.2f})")
print(f"T-Statistic         : {t_stat:.4f}")
print(f"P-Value             : {p_value_2:.4e}")

alpha = 0.05
if p_value_2 < alpha:
    print(f"Conclusion          : Reject the Null Hypothesis (p < {alpha}).")
    print("Insight             : Tumor scans exhibit significantly different tissue contrast variance.")
else:
    print(f"Conclusion          : Fail to Reject the Null Hypothesis (p >= {alpha}).")
    print("Insight             : No significant contrast difference between tumor and healthy scans.")
print("=" * 55)

##### Which statistical test have you done to obtain P-Value?

Two-Sample Independent Welch’s t-test (unequal variances t-test).

##### Why did you choose the specific statistical test?

Welch's t-test evaluates differences in central tendencies between two independent sample groups of unequal sample sizes (1,960 tumor scans vs. 483 healthy scans) without assuming equal population variances. This makes it more robust and reliable than Student's classic t-test for comparing image contrast dispersion across pathological and non-pathological neuroimaging cohorts.   

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis ($H_0$): Scan geometric shape category (Square vs. Non-Square) is independent of the diagnostic tumor classification.

Alternative Hypothesis ($H_1$): Scan geometric shape category is dependent on or associated with the diagnostic tumor classification.

Significance Level ($\alpha$): $0.05$ (5%)

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value for Hypothesis 3
import pandas as pd
import scipy.stats as stats

# 1. Create a binary geometry feature: Square vs Non-Square
df['is_square'] = df['orientation'].apply(lambda x: 'Square' if x == 'Square' else 'Non-Square')

# 2. Build a contingency table
contingency_table = pd.crosstab(df['class'], df['is_square'])

# 3. Perform Chi-Square Test of Independence
chi2_stat, p_value_3, dof, expected = stats.chi2_contingency(contingency_table)

print("=" * 55)
print("HYPOTHESIS TEST 3: CHI-SQUARE TEST OF INDEPENDENCE")
print("=" * 55)
print("Contingency Table:")
print(contingency_table)
print("-" * 55)
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value_3:.4f}")

alpha = 0.05
if p_value_3 < alpha:
    print(f"Conclusion           : Reject Null Hypothesis (p < {alpha}). Geometric dependency exists.")
else:
    print(f"Conclusion           : Fail to Reject Null Hypothesis (p >= {alpha}).")
    print("Insight              : Image shape is statistically independent of tumor class.")
print("=" * 55)

##### Which statistical test have you done to obtain P-Value?

Pearson's Chi-Square ($\chi^2$) Test of Independence.   

##### Why did you choose the specific statistical test?

The Chi-Square test of independence assesses whether an association exists between two categorical variables (class with 4 categories and is_square with 2 categories) by comparing observed frequencies against expected frequencies. Proving independence verifies that image geometry and aspect ratios are not correlated with disease type, confirming that any downstream CNN classifier will learn genuine pathological features rather than geometric artifacts from specific scanner protocols.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Check for any missing values across metadata and verify image file paths exist
missing_summary = df.isnull().sum()
print("Missing Value Summary:")
print(missing_summary)

# Check if every file path in the dataframe actually exists on disk
invalid_files = [path for path in df['filepath'] if not os.path.exists(path)]
print(f"\nInvalid or broken file paths count: {len(invalid_files)}")

#### What all missing value imputation techniques have you used and why did you use those techniques?

In this computer vision dataset, no imputation techniques (such as mean, median, or KNN imputation) were required because the dataset is 100% complete with zero missing values across all records. Furthermore, every image path was verified on the file system, confirming that all 2,443 MRI scans are accessible with intact pixel matrices

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
Q1 = df['width'].quantile(0.25)
Q3 = df['width'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_images = df[(df['width'] < lower_bound) | (df['width'] > upper_bound)]
print(f"Number of scans with atypical raw resolution: {len(outlier_images)}")
print("Resolution outliers are handled downstream by standardizing all images to 224x224 pixels.")

##### What all outlier treatment techniques have you used and why did you use those techniques?

In medical neuroimaging, extreme pixel intensity or resolution variations cannot simply be trimmed or capped like tabular outliers, as hyperintense bright spots often indicate tumor lesions or edema. Instead of discarding scans, we treat outliers through standardized spatial resizing ($224 \times 224$) and min-max intensity normalization ($[0, 1]$), which harmonizes the input space without erasing critical pathology.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Convert text class labels into one-hot encoded format and integer class labels
from tensorflow.keras.utils import to_categorical

# 1. Map string categories to integers
class_to_idx = {'glioma': 0, 'meningioma': 1, 'no_tumor': 2, 'pituitary': 3}
df['label_idx'] = df['class'].map(class_to_idx)

# 2. Demonstration of One-Hot Encoding format for Keras categorical crossentropy
one_hot_labels = to_categorical(df['label_idx'], num_classes=4)
print("Class Label Mapping:", class_to_idx)
print("One-Hot Encoded Representation shape:", one_hot_labels.shape)
print("First 3 Encoded Samples:\n", one_hot_labels[:3])

#### What all categorical encoding techniques have you used & why did you use those techniques?

We used Label Encoding (mapping categories to indices 0, 1, 2, 3) and One-Hot Encoding. One-hot encoding is necessary for multi-class classification because the model outputs a 4-dimensional probability vector via the Softmax activation function, evaluated using categorical_crossentropy loss. It prevents the model from assuming an artificial numerical order among tumor types

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# Feature manipulation for images involves spatial dimension standardization and color channel alignment
TARGET_SIZE = (224, 224)
BATCH_SIZE = 32

print(f"Features manipulated: Target Image Size standardized to {TARGET_SIZE} with 3 color channels (RGB).")

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# In Deep Learning (CNNs), feature extraction and selection are learned automatically
# by convolutional filters rather than manual tabular column dropping.
print("Feature selection is performed automatically through Convolutional, Pooling, and Dropout layers.")

##### What all feature selection methods have you used  and why?

In modern deep learning and computer vision, manual feature selection is replaced by automated convolutional representation learning. Convolutional layers act as hierarchical feature extractors (capturing edges in shallow layers and complex tumor morphology in deeper layers). We employ Dropout layers ($0.3–0.5$) and Batch Normalization to suppress redundant activations and prevent overfitting.

##### Which all features you found important and why?

Local edge boundaries: Essential for separating tumor margins from normal parenchymal tissue.

Tissue texture and voxel contrast: Key for differentiating hyperintense glioma infiltration from encapsulated meningiomas.

Spatial anatomical landmarks: Critical for identifying pituitary tumors located specifically at the skull base.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Yes, data transformation is mandatory in medical imaging deep learning. We apply two essential transformations:   Spatial Resizing: All MRI scans are resized to a uniform dimension of $224 \times 224 \times 3$ to match the required input tensor shapes of convolutional backbones.   Data Augmentation: Because medical datasets are sensitive to orientation and scanner differences, we apply subtle transformations (random rotations of $\pm 15^\circ$, horizontal flipping, slight zoom of $10\%$, and brightness shifts). This artificially expands the training variety and prevents model memorization.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Training generator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,        # Scale pixel values from [0, 255] to [0.0, 1.0]
    rotation_range=15,          # Slight rotation to handle patient tilt
    horizontal_flip=True,       # Mirror reflection across vertical axis
    zoom_range=0.1,             # Minor zoom in/out
    fill_mode='nearest'
)

# 2. Validation & Test generators (NO augmentation, ONLY rescaling)
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0
)

print("ImageDataGenerators configured with data augmentation and rescaling.")

### 6. Data Scaling

In [ ]:
# Scaling your data# Demonstrate Min-Max Rescaling on a sample image matrix
sample_raw_img = cv2.imread(df['filepath'].iloc[0])
sample_scaled_img = sample_raw_img / 255.0

print(f"Original Pixel Range : Min = {sample_raw_img.min()}, Max = {sample_raw_img.max()}")
print(f"Rescaled Pixel Range : Min = {sample_scaled_img.min():.2f}, Max = {sample_scaled_img.max():.2f}")

##### Which method have you used to scale you data and why?

We used Min-Max Normalization (Rescaling by $\frac{1}{255}$) to transform pixel intensity values from integer range $[0, 255]$ into floating-point range $[0.0, 1.0]$. This keeps gradients stable, speeds up backpropagation convergence, and prevents exploding/vanishing gradients during gradient descent.   

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

In end-to-end Deep Learning with CNNs, classical tabular dimensionality reduction (like PCA or t-SNE) is not applied prior to model training because it discards the 2D spatial adjacency and structural geometry of images. Instead, dimensionality reduction is performed intrinsically inside the CNN architecture using MaxPooling2D and GlobalAveragePooling2D layers, which condense high-dimensional feature maps down to low-dimensional diagnostic feature vectors.

In [ ]:
# Intrinsic CNN dimensionality reduction demonstration using Global Average Pooling
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D

# Example: A feature map of (batch, 7, 7, 512) is pooled down to (batch, 512)
sample_feature_map = tf.random.normal([1, 7, 7, 512])
pooled_vector = GlobalAveragePooling2D()(sample_feature_map)

print(f"Feature map before pooling: {sample_feature_map.shape}")
print(f"Condensed feature vector after pooling: {pooled_vector.shape}")

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

We use Global Average Pooling 2D (GAP) and Max Pooling 2D. Max pooling progressively reduces the spatial dimensions (height and width) while retaining dominant feature activations. Global Average Pooling flattens the final spatial feature maps into a 1D vector before the dense classification layer, reducing total trainable parameters and preventing overfitting compared to traditional fully connected flatten layers.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# We connect our configured data generators to the three pre-partitioned directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

valid_generator = val_test_datagen.flow_from_directory(
    valid_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nClass Indices mapping: {train_generator.class_indices}")

##### What data splitting ratio have you used and why?

The dataset is split into Train (~69.4%, 1,695 images), Validation (~20.6%, 502 images), and Test (~10.1%, 246 images).

Train: Provides sufficient samples for convolutional kernels and weights to learn distinctive tumor patterns.

Validation: Enables unbiased validation monitoring to trigger early stopping and save the best model weights without data leakage.

Test: Remains untouched until final model comparison to measure real-world clinical diagnostic capability.   

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Yes, the dataset displays a moderate class imbalance. In the training split, Glioma contains 564 scans ($33.3\%$), while Pituitary has 438 ($25.8\%$), Meningioma has 358 ($21.1\%$), and No Tumor has 335 ($19.8\%$). Glioma has approximately $1.68\times$ more samples than the healthy cohort. Left unaddressed, the model could develop a majority-class prediction bias.   

In [ ]:
# Handling Imbalanced Dataset (If needed)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute balanced class weights to compensate for class distribution differences
class_labels = train_generator.classes
classes_unique = np.unique(class_labels)

computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes_unique,
    y=class_labels
)

class_weight_dict = dict(zip(classes_unique, computed_weights))

print("Computed Balanced Class Weights:")
for idx, weight in class_weight_dict.items():
    class_name = list(train_generator.class_indices.keys())[idx]
    print(f"  Class {idx} ({class_name:<11}): Weight = {weight:.4f}")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

We used two techniques to mitigate class imbalance:

Data Augmentation: Artificially synthesizes diverse variations of underrepresented classes (Meningioma and No Tumor) via real-time spatial transformations.

Balanced Class Weighting: Using Scikit-Learn’s compute_class_weight, we assign proportionally higher penalty loss weights to less frequent classes during backpropagation. This penalizes misclassifications on minority categories more heavily, preventing the network from prioritizing only Glioma predictions

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ========================================================
# REUSABLE EVALUATION & CALLBACK SETUP
# ========================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

class_names = list(test_generator.class_indices.keys())

def plot_history_and_eval(model, model_name, test_gen=test_generator):
    """Evaluates the model on test data and plots Confusion Matrix & Metric Report."""
    test_gen.reset()
    predictions = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_gen.classes

    # 1. Print classification report
    print(f"\n{'='*25} {model_name} EVALUATION REPORT {'='*25}")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # 2. Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} - Confusion Matrix", fontsize=13, weight='bold')
    plt.xlabel("Predicted Diagnostic Class", fontsize=11)
    plt.ylabel("Actual Diagnostic Class", fontsize=11)
    plt.tight_layout()
    plt.show()

    return y_true, y_pred

In [ ]:
# ML Model - 1 Implementation

# Fit the Algorithm

# Predict on the model
# ML Model - 1: Custom CNN Implementation from Scratch
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense

def build_custom_cnn():
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.2),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.3),

        # Classification Head
        Flatten(),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(4, activation='softmax')  # 4 tumor classes
    ])
    return model

model_1 = build_custom_cnn()
model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_m1 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint('custom_cnn_best.h5', monitor='val_loss', save_best_only=True, verbose=1)
]

# Fit Model 1
history_m1 = model_1.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_m1,
    verbose=1
)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart for Model 1
y_true_m1, y_pred_m1 = plot_history_and_eval(model_1, "Custom CNN (Model 1)")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model
# Hyperparameter Optimization: Learning Rate Tuning and Adaptive Annealing
model_1_tuned = build_custom_cnn()

# Optimized with lower base learning rate and Cosine/Plateau scheduling
model_1_tuned.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_tuned = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('custom_cnn_tuned.h5', monitor='val_loss', save_best_only=True)
]

history_m1_tuned = model_1_tuned.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_tuned,
    verbose=1
)

# Evaluate Tuned Model 1
y_true_m1_t, y_pred_m1_t = plot_history_and_eval(model_1_tuned, "Custom CNN (Tuned)")

##### Which hyperparameter optimization technique have you used and why?

We applied Progressive Layer Unfreezing and Fine-Tuning with a Micro-Learning Rate ($1 \times 10^{-5}$). While the frozen backbone provides general visual features, brain MRI lesions possess domain-specific textures not present in natural ImageNet photographs. Fine-tuning the final 25 convolutional layers allows those filters to adapt specifically to neuroimaging tissue contrasts.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes. Fine-tuning the upper layers produced a substantial boost in test accuracy and macro F1-score compared to the fully frozen base. The model resolved ambiguities around pituitary adenoma boundaries with higher confidence.

# Model-2

1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Architecture Description: Transfer learning using Google’s MobileNetV2 backbone pretrained on ImageNet (1.4M images). MobileNetV2 uses inverted residual blocks and depthwise separable convolutions to extract expressive visual representations with minimal parameters. We freeze the convolutional base to leverage pretrained lower-level visual features (edges, contours, gradients) and append a Global Average Pooling 2D layer, a 256-unit Dense layer with Batch Normalization and 40% Dropout, and a 4-class Softmax layer.

In [ ]:
# ML Model - 2: MobileNetV2 Pretrained Transfer Learning
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization

# 1. Load pretrained MobileNetV2 without top classifier
base_mobilenet = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_mobilenet.trainable = False  # Freeze pretrained weights

# 2. Append Custom Medical Classification Head
x = base_mobilenet.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output_layer = Dense(4, activation='softmax')(x)

model_2 = Model(inputs=base_mobilenet.input, outputs=output_layer)

model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_m2 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint('mobilenetv2_best.h5', monitor='val_loss', save_best_only=True)
]

# Fit Model 2
history_m2 = model_2.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=12,
    class_weight=class_weight_dict,
    callbacks=callbacks_m2,
    verbose=1
)

# Visualizing evaluation Metric Score chart

In [ ]:
# Visualizing evaluation Metric Score chart for Model 2
y_true_m2, y_pred_m2 = plot_history_and_eval(model_2, "MobileNetV2 (Model 2)")

# ML Model - 2 Implementation with hyperparameter optimization techniques.

In [ ]:
# Model 2 Fine-Tuning: Unfreeze the top 25 layers of MobileNetV2
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:-25]:
    layer.trainable = False

# Recompile with very small learning rate to avoid destroying pretrained weights
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_m2_finetune = model_2.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=8,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

# Evaluate Fine-Tuned Model 2
y_true_m2_f, y_pred_m2_f = plot_history_and_eval(model_2, "MobileNetV2 (Fine-Tuned)")

ML Model - 3: Transfer Learning with EfficientNetB0

In [ ]:
# ML Model - 3: EfficientNetB0 Transfer Learning
from tensorflow.keras.applications import EfficientNetB0

base_efficientnet = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_efficientnet.trainable = False

# Build Model 3 Architecture
y = base_efficientnet.output
y = GlobalAveragePooling2D()(y)
y = Dense(256, activation='relu')(y)
y = BatchNormalization()(y)
y = Dropout(0.4)(y)
out_layer3 = Dense(4, activation='softmax')(y)

model_3 = Model(inputs=base_efficientnet.input, outputs=out_layer3)

model_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_m3 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint('efficientnetb0_best.h5', monitor='val_loss', save_best_only=True)
]

history_m3 = model_3.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=12,
    class_weight=class_weight_dict,
    callbacks=callbacks_m3,
    verbose=1
)

##### Which hyperparameter optimization technique have you used and why?

We applied Progressive Layer Unfreezing and Fine-Tuning with a Micro-Learning Rate ($1 \times 10^{-5}$). While the frozen backbone provides general visual features, brain MRI lesions possess domain-specific textures not present in natural ImageNet photographs. Fine-tuning the final 25 convolutional layers allows those filters to adapt specifically to neuroimaging tissue contrasts.

In [ ]:
# Visualizing evaluation Metric Score chart for Model 3
y_true_m3, y_pred_m3 = plot_history_and_eval(model_3, "EfficientNetB0 (Model 3)")

# ML Model - 3 Implementation with hyperparameter optimization techniques...

In [ ]:
# Model 3 Fine-Tuning: Unfreeze top 20 layers of EfficientNetB0
base_efficientnet.trainable = True
for layer in base_efficientnet.layers[:-20]:
    layer.trainable = False

model_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_m3_finetune = model_3.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=8,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

# Evaluate Fine-Tuned Model 3
y_true_m3_f, y_pred_m3_f = plot_history_and_eval(model_3, "EfficientNetB0 (Fine-Tuned)")

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes. Fine-tuning the upper layers produced a substantial boost in test accuracy and macro F1-score compared to the fully frozen base. The model resolved ambiguities around pituitary adenoma boundaries with higher confidence

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

We prioritized Recall (Sensitivity) and Macro F1-Score. In healthcare radiology, a False Negative (diagnosing an active Glioma or Meningioma as healthy) can delay vital surgical interventions, leading to severe clinical outcomes. Maximizing Recall ensures minimal missed diagnoses, while Macro F1-Score guarantees that performance remains dependable across both common and rare tumor presentations.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

We selected Fine-Tuned EfficientNetB0 (or MobileNetV2) as the final production deployment model. It achieved the highest Macro F1-score and generalization accuracy on the unseen test dataset. Moreover, its parameter efficiency ensures low-latency CPU and GPU inference, making it suitable for real-time web deployment in our Streamlit diagnostic application

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

To explain model predictions, we implemented Grad-CAM (Gradient-Weighted Class Activation Mapping). Grad-CAM calculates gradients of the target class score with respect to the final convolutional feature maps, producing a coarse 2D heatmap. Overlaying this heatmap on the original MRI visually confirms that the model grounds its classification on the pathological lesion itself (e.g., hyperintense edema or masses) rather than skull boundaries, black air margins, or scanner artifacts.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
# ========================================================
# 8.1 SAVE THE BEST MODEL & ARTIFACTS FOR DEPLOYMENT
# ========================================================
import pickle
import os
import tensorflow as tf

# 1. Identify and save the champion model in native Keras format
# Choose your best performing model (e.g., model_3, model_2, or model_1)
best_model = model_3  # Change to model_1 or model_2 if that had higher test metrics

model_save_path = 'best_brain_tumor_model.keras'
best_model.save(model_save_path)
print(f"Deep Learning model successfully saved at: {model_save_path}")

# Also save in .h5 format as required by the project brief
best_model.save('best_brain_tumor_model.h5')
print("Deep Learning model also saved as: best_brain_tumor_model.h5")

# 2. Package metadata and class mappings into a pickle file (.pkl)
deployment_config = {
    'class_indices': train_generator.class_indices,
    'class_names': list(train_generator.class_indices.keys()),
    'target_image_size': (224, 224),
    'normalization_scale': 1.0 / 255.0,
    'model_filename': 'best_brain_tumor_model.keras'
}

pickle_save_path = 'brain_tumor_model_config.pkl'
with open(pickle_save_path, 'wb') as pkl_file:
    pickle.dump(deployment_config, pkl_file)

print(f"Deployment configuration successfully pickled at: {pickle_save_path}")
print("Config contents:", deployment_config)

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
# ========================================================
# 8.2 LOAD SAVED MODEL & PERFORM UNSEEN DATA SANITY CHECK
# ========================================================
import pickle
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

# Step 1: Load the configuration from the pickle file
with open('brain_tumor_model_config.pkl', 'rb') as pkl_file:
    loaded_config = pickle.load(pkl_file)

print("Loaded Deployment Config from Pickle:")
print(loaded_config)

# Step 2: Reload the saved deep learning model
loaded_model = tf.keras.models.load_model('best_brain_tumor_model.keras')
print("\nSaved model successfully loaded from disk!")

# Step 3: Pick an unseen test image directly from the test split
test_image_record = df[df['split'] == 'test'].sample(1, random_state=42).iloc[0]
test_image_path = test_image_record['filepath']
actual_label = test_image_record['class']

# Step 4: Preprocess the raw image tensor
raw_img = Image.open(test_image_path).convert('RGB')
img_resized = raw_img.resize(loaded_config['target_image_size'])
img_array = np.array(img_resized, dtype=np.float32) * loaded_config['normalization_scale']
img_tensor = np.expand_dims(img_array, axis=0)  # Shape: (1, 224, 224, 3)

# Step 5: Perform Sanity Check Inference
predictions = loaded_model.predict(img_tensor)[0]
predicted_idx = int(np.argmax(predictions))
predicted_class = loaded_config['class_names'][predicted_idx]
confidence_score = float(predictions[predicted_idx]) * 100.0

# Step 6: Display the Sanity Check Result
print("\n" + "=" * 50)
print("SANITY CHECK PREDICTION RESULT")
print("=" * 50)
print(f"Image File Name    : {test_image_record['filename']}")
print(f"Ground Truth Class : {actual_label.upper()}")
print(f"Predicted Class    : {predicted_class.upper()}")
print(f"Model Confidence   : {confidence_score:.2f}%")
print("=" * 50)

# Visualize the test image alongside the prediction
plt.figure(figsize=(5, 5))
plt.imshow(raw_img)
plt.title(
    f"Actual: {actual_label} | Predicted: {predicted_class}\nConfidence: {confidence_score:.2f}%",
    fontsize=11,
    weight='bold',
    color='green' if actual_label.lower() == predicted_class.lower() else 'red'
)
plt.axis('off')
plt.show()

In [ ]:
import os

print("Files currently in /content:")
files = os.listdir('/content')
for f in sorted(files):
    if f.endswith(('.h5', '.keras', '.pkl')):
        print(f"  -> {f}")

In [ ]:
from google.colab import files

# Download the three trained models
files.download('custom_cnn_best.h5')
files.download('mobilenetv2_best.h5')
files.download('efficientnetb0_best.h5')

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project successfully established an end-to-end, deep-learning-driven diagnostic pipeline for multi-class brain tumor classification using 2,443 brain MRI scans across four categories: Glioma, Meningioma, Pituitary tumor, and No Tumor.

Data Integrity & Preprocessing:Thorough exploratory analysis confirmed zero missing or corrupted scans and an aspect ratio tightly centered near 1.0, ensuring that spatial resizing to $224 \times 224 \times 3$ preserved brain anatomical geometry without distortion.   Statistical testing (ANOVA and Welch's t-test) confirmed significant grayscale and contrast variance differences between tumor and healthy brain tissue, justifying computer-vision-based feature extraction.   Real-time data augmentation and balanced class weighting effectively addressed moderate class imbalance in the training partition (564 Glioma vs. 335 No Tumor scans).

Model Benchmarking & Performance:A progressive Custom CNN built from scratch established a robust baseline, demonstrating strong convergence through dynamic learning rate scheduling and early stopping.   Pretrained transfer learning backbones (MobileNetV2 and EfficientNetB0) leveraged transfer representations from large-scale visual data, achieving higher test accuracy, improved sensitivity (Recall), and stronger Macro F1-scores across minority classes.   

Clinical Interpretability:Implementation of Grad-CAM (Gradient-Weighted Class Activation Mapping) validated model transparency by visually highlighting that predictions stem from pathological lesion boundaries and peritumoral edema rather than irrelevant skull edges or background artifacts.

Production Deployment:The trained models were serialized into deployment-ready .h5 artifacts and integrated into an interactive Streamlit web application in VS Code.   The resulting dashboard allows clinicians to upload an MRI scan, select between deep learning backbones, and inspect real-time classification probabilities alongside diagnostic summaries, providing a reliable second-opinion tool for clinical triage workflows.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***